# Supervised Learning: Algorithmic Foundations & Comparative Lab

Supervised learning algorithms infer a functional mapping $f: \mathcal{X} \to \mathcal{Y}$ from labeled training pairs $\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^N$. This lab implements and evaluates core regression and classification families, comparing closed-form and iterative optimization regimes.

### Table of Contents
1. [Linear, Ridge & Lasso Regression: Regularization Dynamics](#1-linear-ridge--lasso-regression-regularization-dynamics)
2. [Logistic Regression & Probabilistic Calibration](#2-logistic-regression--probabilistic-calibration)
3. [Decision Trees vs. Random Forests: Variance Reduction](#3-decision-trees-vs-random-forests-variance-reduction)
4. [Support Vector Machines (SVM): Maximum Margin Separators](#4-support-vector-machines-svm-maximum-margin-separators)
5. [Validation & Performance Benchmarks against Scikit-Learn](#5-validation--performance-benchmarks-against-scikit-learn)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_regression, make_moons
from sklearn.linear_model import Ridge, Lasso, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, accuracy_score

sns.set_theme(style="whitegrid")
print("Loaded Supervised Learning interactive environment.")

## 1. Linear, Ridge & Lasso Regression: Regularization Dynamics
- **OLS**: $\min_w \|y - Xw\|_2^2$
- **Ridge ($L_2$)**: $\min_w \|y - Xw\|_2^2 + \alpha \|w\|_2^2$ (smooth shrinkage)
- **Lasso ($L_1$)**: $\min_w \frac{1}{2n} \|y - Xw\|_2^2 + \alpha \|w\|_1$ (sparse feature selection)

In [ ]:
# Generate dataset with correlated & uninformative features
np.random.seed(42)
X, y, coef = make_regression(n_samples=100, n_features=10, n_informative=3, noise=5.0, coef=True, random_state=42)

alphas = np.logspace(-2, 2, 50)
ridge_weights = []
lasso_weights = []

for a in alphas:
    r = Ridge(alpha=a).fit(X, y)
    l = Lasso(alpha=a, max_iter=2000).fit(X, y)
    ridge_weights.append(r.coef_)
    lasso_weights.append(l.coef_)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(alphas, ridge_weights)
axes[0].set_xscale("log")
axes[0].set_title("Ridge Regularization Path (L2 Shrinkage)", fontweight="bold")
axes[0].set_xlabel("Alpha")
axes[0].set_ylabel("Coefficients")

axes[1].plot(alphas, lasso_weights)
axes[1].set_xscale("log")
axes[1].set_title("Lasso Regularization Path (L1 Sparsity)", fontweight="bold")
axes[1].set_xlabel("Alpha")
axes[1].set_ylabel("Coefficients")
plt.tight_layout()
plt.show()

## 2. Logistic Regression & Decision Boundaries
Modeling class posterior $P(y=1 | \mathbf{x}) = \sigma(\mathbf{w}^T \mathbf{x} + b)$ minimizing cross-entropy loss.

In [ ]:
np.random.seed(0)
X_bin, y_bin = make_moons(n_samples=200, noise=0.25, random_state=0)
log_reg = LogisticRegression().fit(X_bin, y_bin)

# Visualize linear decision boundary
x_min, x_max = X_bin[:, 0].min() - 0.5, X_bin[:, 0].max() + 0.5
y_min, y_max = X_bin[:, 1].min() - 0.5, X_bin[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 150), np.linspace(y_min, y_max, 150))
Z = log_reg.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

plt.figure(figsize=(7, 5))
plt.contourf(xx, yy, Z, levels=20, cmap="RdBu", alpha=0.6)
plt.scatter(X_bin[:, 0], X_bin[:, 1], c=y_bin, cmap="RdBu", edgecolors="k", s=40)
plt.title(f"Logistic Regression Decision Surface (Accuracy: {accuracy_score(y_bin, log_reg.predict(X_bin)):.2%})", fontweight="bold")
plt.xlabel("Feature X1")
plt.ylabel("Feature X2")
plt.tight_layout()
plt.show()

## 3. Decision Trees vs. Random Forests: Variance Reduction
Single decision trees have high variance and produce sharp axis-aligned partitions. Ensembling via bootstrap aggregation and feature subsampling (Random Forest) smooths decision boundaries.

In [ ]:
tree = DecisionTreeClassifier(max_depth=None, random_state=42).fit(X_bin, y_bin)
forest = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42).fit(X_bin, y_bin)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, model, name in zip(axes, [tree, forest], ["Single Deep Decision Tree", "Random Forest (100 Trees)"]):
    Z_m = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z_m, cmap="RdBu", alpha=0.5)
    ax.scatter(X_bin[:, 0], X_bin[:, 1], c=y_bin, cmap="RdBu", edgecolors="k", s=35)
    ax.set_title(f"{name}\nAccuracy: {accuracy_score(y_bin, model.predict(X_bin)):.2%}", fontweight="bold")
    ax.set_xlabel("X1")
    ax.set_ylabel("X2")
plt.tight_layout()
plt.show()

## 4. Support Vector Machines (SVM): Maximum Margin Separators
SVM maximizes the geometric margin $\frac{2}{\|\mathbf{w}\|}$ separating support vectors.

In [ ]:
from sklearn.svm import SVC

svm_rbf = SVC(kernel="rbf", C=1.0, gamma=2.0).fit(X_bin, y_bin)
Z_svm = svm_rbf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(7, 5))
plt.contour(xx, yy, Z_svm, levels=[-1, 0, 1], linestyles=["--", "-", "--"], colors="black")
plt.contourf(xx, yy, Z_svm, levels=20, cmap="coolwarm", alpha=0.4)
plt.scatter(X_bin[:, 0], X_bin[:, 1], c=y_bin, cmap="coolwarm", edgecolors="k", s=40)
plt.scatter(svm_rbf.support_vectors_[:, 0], svm_rbf.support_vectors_[:, 1], s=100, facecolors="none", edgecolors="yellow", lw=1.5, label="Support Vectors")
plt.title(f"Non-Linear SVM with RBF Kernel (Support Vectors: {len(svm_rbf.support_vectors_)})", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()